# PlanetSWE cache for Colab / Kaggle

The full PlanetSWE (The Well, about 190 GB) does not fit on Colab or Kaggle, and reading it from the Hugging Face
Hub one task at a time is far too slow. This notebook streams it **once** and keeps only what the tasks need,
in a cache of a few GB that `swe_np_comparison.ipynb` loads (`SWE_CACHE_DIR`):
- the 20 sensors and a fixed pool of `POOL_SIZE` grid points (drawn in proportion to their cell area), with all
  three variables (h, u_theta, u_phi);
- only at every 5th time step, ending at the last one (the readings of a task are 5 steps apart, so they all
  fall on that lattice), which also means that only about a fifth of every file is transferred.

Per trajectory the cache takes about 20 MB with 8192 points (1.6 GB per file on the Hub). The tasks then draw
their targets from the pool.

**How to run it**
1. **Colab**: a CPU runtime is enough. The cache is written to the Drive folder `CACHE_DIR`.
   **Kaggle**: a CPU notebook with *Internet* on. The cache is written to `/kaggle/working/swe_cache`; afterwards
   use *Save Version* (Save & Run All) and then *New Dataset* in the Output tab, to use it in the training notebook.
2. If you have the sensor file `SWE_sensors_coordinates.pt`, put it in `CACHE_DIR`'s parent (Drive: the folder of
   the cache; Kaggle: an input dataset) or next to `swe_data.py`. Without it, 20 evenly spread sensors are used.
3. The job is **resumable**: finished trajectories are kept, and running the build cell again only does the
   rest (use this if a session ends before it finishes). How long it takes depends on the connection of the
   machine (the printed time after each trajectory tells), and `FILES_PER_SPLIT` limits the number of trajectories.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

# Edit these if needed.
REPO_URL, REPO_BRANCH = "https://github.com/Paolofox01/NP.git", "main"
USE_DRIVE = True                                      # Colab: mount Google Drive and keep the cache there
DRIVE_CACHE_DIR = "/content/drive/MyDrive/swe_cache"


def sh(command):
    """Runs a shell command, showing its output in the cell as it comes."""
    process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="")
    if process.wait():
        raise RuntimeError(f"Command failed (exit code {process.returncode}): {command}")


if IN_COLAB or IN_KAGGLE:
    work = Path("/content" if IN_COLAB else "/kaggle/working")
    repo = work / "NP"
    if repo.exists():
        sh(f"git -C {repo} pull --ff-only")
    else:
        sh(f"git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {repo}")
    sys.path.insert(0, str(repo / "swe"))
    os.chdir(repo / "swe")
    if IN_COLAB and USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        CACHE_DIR = Path(DRIVE_CACHE_DIR)
    else:
        CACHE_DIR = work / "swe_cache"
    sensor_candidates = [CACHE_DIR.parent / "SWE_sensors_coordinates.pt", repo / "swe" / "SWE_sensors_coordinates.pt"]
    if Path("/kaggle/input").exists():
        sensor_candidates += sorted(Path("/kaggle/input").rglob("SWE_sensors_coordinates.pt"))
else:
    sys.path.insert(0, str(Path.cwd()))                # the notebook's folder, with swe_data.py
    CACHE_DIR = Path("swe_cache")
    sensor_candidates = [Path("SWE_sensors_coordinates.pt")]

# Streaming needs these (already there on Colab and Kaggle, usually).
try:
    import h5py, huggingface_hub
except ImportError:
    sh(f"{sys.executable} -m pip install -q h5py huggingface_hub")

SENSOR_FILE = next((p for p in sensor_candidates if p.exists()), None)
print(f"Cache: {CACHE_DIR}\nSensor file: {SENSOR_FILE or 'none: 20 evenly spread sensors'}")

## Build the cache
`FILES_PER_SPLIT`: how many trajectories of each split to use, evenly spaced (`None` = all: 96 train, 12 valid,
12 test). `WORKERS` trajectories are streamed in parallel. A Hugging Face token (`HF_TOKEN` in the environment)
raises the download limits, but it is not required.

In [ ]:
from swe_data import build_cache, load_cache, SENSOR_FILE as DEFAULT_SENSOR_FILE

POOL_SIZE = 8192                      # grid points kept per time step (besides the sensors)
FILES_PER_SPLIT = None                # e.g. {"train": 24, "valid": 6, "test": 6} to transfer less
WORKERS = 8                           # trajectories streamed in parallel
SEED = 0                              # draws the pool of points

build_cache(CACHE_DIR, pool_size=POOL_SIZE, files_per_split=FILES_PER_SPLIT, workers=WORKERS, seed=SEED,
            sensor_file=SENSOR_FILE or DEFAULT_SENSOR_FILE)

## Check the cache

In [ ]:
data = load_cache(CACHE_DIR)
size = sum(f.stat().st_size for f in Path(CACHE_DIR).rglob("*") if f.is_file())
print(f"{size / 1e9:.2f} GB on disk")
for split, values in data.trajectories.items():
    print(f"{split}: {tuple(values.shape)}  (trajectories, times, points = sensors + pool, variables)")
print("sensors:", len(data.sensors), "| mean:", data.y_mean.tolist(), "| std:", data.y_std.tolist())